# 16.1 最慢或最佔空間的是哪裡？


模型跑完用了兩秒，究竟慢在注意力、特徵處理，還是程式第一次初始化？只看總時間，像只知道一餐準備了多久，卻不知道該改善哪個步驟。開始改效率以前，先量出成本的位置。本章標題的BKM指Best Known Method，也就是在明確條件下目前已知的做法；條件變了，應重新量測，不能把某個選項名稱當成速度保證。

前置是[一層模型的組成](https://birdhackor.github.io/tiny-perceptron-vlm/4.5.html)與[矩陣使用量不等於耗時](https://birdhackor.github.io/tiny-perceptron-vlm/15.11.html)。這裡使用已備妥的TinyLM小型文字模型：輸入token ID，經查表、注意力與FFN後，替每個位置輸出詞表分數。profiler（效能剖析器）會記錄各運算的時間與呼叫次數，幫我們找到下一個值得研究的瓶頸。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch.profiler import profile, ProfilerActivity
from tiny_perceptron.model import TinyLM, ModelConfig

torch.set_num_threads(1)
model = TinyLM(ModelConfig(width=8)).eval()
ids = torch.tensor([[1, 2, 3]])
with torch.no_grad():
    for _ in range(3):
        model(ids)
    with profile(activities=[ProfilerActivity.CPU]) as p:
        for _ in range(5):
            result = model(ids)
print("分數形狀", tuple(result["logits"].shape))
print(p.key_averages().table(sort_by="self_cpu_time_total", row_limit=5))
print("權重bytes", sum(v.numel() * v.element_size() for v in model.parameters()))

輸入一列三個ID，預設詞表264項，第一行應為 `(1,3,264)`，可確認確實執行了整個模型而非空迴圈。先跑三次暖機，避免把第一次初始化當成穩定成本；再記錄五次向前計算，`key_averages`依運算名稱彙整。這裡只開CPU活動，不會替你量到GPU工作。

表中的Self CPU表示運算自身花的時間，不包含子運算；CPU total還包括它呼叫的子運算。按Self排序有助找直接耗時的工作，但不能把全部total列加起來，因為父子可能重複計數。`# of Calls`是記錄期間呼叫次數：有些線性層在模型裡出現多次，五輪後次數自然比5大。`aten::`前綴是PyTorch底層運算名稱，例如矩陣乘法與softmax；在此TinyLM中，`aten::index_select`對應字與位置的查表，`aten::softmax`或`aten::_softmax`對應注意力分數轉權重，`aten::gelu`則對應FFN中的啟動函數。這樣才能由名稱追到模型哪一段；表的排序可能隨本機量測改變。

最後一行的`numel()`回報參數表有幾個元素，`element_size()`回報每個元素占幾bytes，兩者相乘才是該表的數字bytes，再沿所有參數表相加。此設定6104個FP32參數、每個4bytes，應得24416bytes；它與表中的微秒是不同單位。執行時還有中間特徵、生成用的cache；訓練更有梯度與優化器狀態，所以權重大小不能代表最高記憶體占用。profiler本身也會增加開銷，主要用於定位，總耗時應另以重複計時核對。

本章另完成一組L4實驗，使用PyTorch 2.14.1+cu126、FP32、關閉TF32。TF32是支援的GPU處理FP32矩陣乘法時可用的一種模式：資料仍以FP32儲存，乘法卻可使用較少的精細位數，再以FP32累加；它會改變計算精度，也可能影響速度，因此本輪明記關閉，固定這個計算條件。前置是[T.4](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)的直接屬性SFT模型與45／5／10題家族切分；原模型最後檢查答對5／10，不能因它可載入就當成全對。這次把模型改為四個Query頭，MHA與一個KV頭的組別各短續訓100次，再從MHA副本比較不同更新方式。形狀改動與留出結果在[16.4](https://birdhackor.github.io/tiny-perceptron-vlm/16.4.html)，重跑入口在[T.8](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.8)。

推論各先暖機三次，再同步量九次、取中位數。中位數先把時間由小到大排序：九次採樣取第5個；若留下偶數次採樣，則取中間兩個的平均，例如 `[1,2,3,20]` 的中位數是 `(2+3)/2=2.5`。一般平均數則把全部採樣相加再除以次數。訓練另略過前三步取更新中位數，不能把兩種時間混在同一欄。

這份報告中的六支主要訓練——MHA與GQA[注意力頭數比較](https://birdhackor.github.io/tiny-perceptron-vlm/16.4.html)、[一般更新與梯度累積](https://birdhackor.github.io/tiny-perceptron-vlm/16.6.html)、[反向重算](https://birdhackor.github.io/tiny-perceptron-vlm/16.10.html)及SDPA[注意力API](https://birdhackor.github.io/tiny-perceptron-vlm/16.8.html)——各自重設PyTorch管理tensor記憶體的CUDA配置器峰值，保存開始前已配置、該段峰值與新增量。[短文件裝填](https://birdhackor.github.io/tiny-perceptron-vlm/16.5.html)的padded與packed兩支更新沒有保存這三個記憶體欄位，因此這份報告不能拿它們比較逐支線峰值。

上述記憶體數字只計PyTorch在目前訓練程序中交給tensor使用的GPU空間。driver是GPU驅動程式，也就是讓作業系統與GPU溝通的軟體；它本身的用量不在這個數字中。PyTorch留著準備再用、當時沒有交給tensor的空間，稱為未使用的reserved空間，也不在其中。本輪另用Inductor這個PyTorch編譯工具，把運算轉成可執行的程式；這項工作由另一個獨立程序執行，該程序的用量沒有算進上述訓練峰值。

後面以MiB列量，1 MiB是2²⁰ bytes。這個範圍可對照[2.14的max_memory_allocated文件](https://docs.pytorch.org/docs/2.14/generated/torch.cuda.max_memory_allocated.html)，不能把外層最後一次reset後的數字稱為整組實驗峰值。完整數字與原始採樣在[efficiency實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/efficiency.json)。

練習把ids改成 `torch.arange(1,13)[None]`，也就是一段十二個token。先預測分數形狀變成 `(1,12,264)`、權重bytes不變，再重跑比較前五項時間。短CPU例子可能排序不穩定，應多跑幾輪；你要找的是隨序列長度變大的成本，而不是記住某台機器某次的第一名。
